# Resynthesize an events table

`conga_multidft` holds 87 steady partials from four analysis frames of a
conga sound. This notebook picks a sample rate, renders the table with
`synthesize`, compares phase modes and fades at the frame edges, then lets
partials fold on purpose at a low rate. Read `df_pitch_stretch` first if
events tables are new.

In [ ]:
from pathlib import Path
import numpy as np
from scipy import signal
import audiospylt as asp

## Events table

Every row keeps one frequency (`freq_start == freq_stop`). The frames start at
0.03, 0.10, 0.20 and 0.40 s, and the last row ends at 0.75 s.

In [ ]:
events = asp.read_table("../data/tables/conga_multidft.tsv")
print(asp.infer_kind(events), f"{len(events)} rows")
print(f"steady: {(events.freq_start == events.freq_stop).all()}")
print(f"freq {events.freq_start.min():.1f}–{events.freq_start.max():.1f} Hz")
print("rows per frame start:", events.groupby("time_start").size().to_dict())
print(f"last end {events.time_stop.max():.2f} s")
asp.nb.show(asp.viz.plot_events_3d(events))

## Sample rate

Every partial must stay below half the sample rate (the Nyquist frequency), so
the rate must be strictly above twice the highest frequency, 6342.9 Hz here.
The renders below use 44100 Hz.

In [ ]:
f_max = events.freq_start.max()
print(f"2 × {f_max:.1f} Hz = {2 * f_max:.1f} Hz")
sr = 44100

## Default synthesis

By default `synthesize` uses `phase="continuous"`: a row that starts where
another stops, within `link_cents` (a semitone) of its frequency, takes over
that row's phase, one to one and nearest frequency first. An amplitude step at
such a link is smoothed over `link_ramp` (0.005 s), and each voice starts and
ends with a `fade` of the same length. The 87 partials add up to a peak above
1, and `nb.play` refuses to clip, so play with `normalize=True`.

In [ ]:
audio = asp.synthesize(events, sr=sr)
print(f"{len(audio) / sr:.3f} s, peak {np.max(np.abs(audio)):.2f}")
asp.nb.show(asp.viz.plot_waveform(audio, sr))
asp.nb.show(asp.viz.plot_spectrogram(asp.spectrogram(audio, sr), audio=audio))
asp.nb.play(audio, sr, normalize=True)

## Phase modes and link ramps

`phase="reset"` starts every row at phase 0, and `phase="absolute"` takes each
row's phase from the clock (2π·f·t at its start). In both, every row is its own
voice with its own fades, so nothing links. `link_ramp=0` keeps continuous
phase but leaves the amplitude steps at the links hard. The function below
measures clicks: the energy above 6000 Hz within 0.002 s of each frame edge, in
dB re the render's RMS. `help(asp.synthesize)` lists `link_cents` and the
other options.

In [ ]:
highpass = signal.butter(8, 6000, "high", fs=sr, output="sos")

def clicks(a, edges=(0.10, 0.20, 0.40)):
    hf = signal.sosfiltfilt(highpass, a)
    db = [10 * np.log10(np.mean(asp.trim(hf, sr, t - 0.002, t + 0.002) ** 2) / np.mean(a**2))
          for t in edges]
    return [round(d) for d in db]

Before you run the cell: which render clicks, and which dips in level at the
frame edges? The plot overlays `"continuous"` and `"reset"` around the edge
at 0.20 s.

In [ ]:
renders = {
    "continuous": asp.synthesize(events, sr=sr),
    "reset": asp.synthesize(events, sr=sr, phase="reset"),
    "absolute": asp.synthesize(events, sr=sr, phase="absolute"),
    "link_ramp=0": asp.synthesize(events, sr=sr, link_ramp=0),
}
for label, a in renders.items():
    print(f"{label}: {clicks(a)} dB at 0.10 / 0.20 / 0.40 s")
    asp.nb.play(a, sr, normalize=True)
zoomed = ["continuous", "reset"]
edge = np.vstack([asp.trim(renders[label], sr, 0.19, 0.21) for label in zoomed])
asp.nb.show(asp.viz.plot_waveform(edge, sr, time_offset=0.19, channel_labels=zoomed))

Only `link_ramp=0` clicks: −31 dB at 0.10 s, against −83 dB with the default
ramp. `"reset"` and `"absolute"` stay at −97 dB or below, but each row fades
out before the edge and the next fades in after it, so the level dips at every
frame edge, a soft pulse: in the zoom `"reset"` falls almost silent around
0.20 s while `"continuous"` carries on. `"absolute"` joins
without a dip only where a row repeats the previous frequency and `fade=0`.

## Fades

`fade` shapes the start and end of every voice. The cell adds the first frame
edge, 0.03 s, where 51 rows start together.

In [ ]:
for fade in (0.005, 0):
    a = asp.synthesize(events, sr=sr, fade=fade)
    print(f"fade={fade}: {clicks(a, (0.03, 0.10, 0.20, 0.40))} dB at 0.03 / 0.10 / 0.20 / 0.40 s")
    asp.nb.play(a, sr, normalize=True)

With `fade=0` the 51 rows start at full amplitude: −25 dB at 0.03 s, against
−127 dB with the default fade. Every edge where rows end clicks too (−34 dB at
0.10 s).

## Aliasing on purpose

At `sr=4000` the Nyquist frequency is 2000 Hz and 13 rows lie above it. By
default (`nyquist="drop"`) their samples are silent and `synthesize` warns.
With `nyquist="alias"` they fold back as sampling predicts: a partial at f
sounds at |f − k·sr|, with k the whole number nearest to f / sr. Before you
run the cell: where does the top partial, 3171.4 Hz, land?

In [ ]:
sr_low = 4000
high = events[events.freq_start > sr_low / 2]
print(f"{len(high)} rows above {sr_low / 2:g} Hz")
folded = (high.freq_start - sr_low * np.round(high.freq_start / sr_low)).abs()
high.assign(folded=folded)[["freq_start", "folded"]].round(1)

3171.4 Hz lands at 828.6 Hz (k = 1). Render the table at 4000 Hz both ways
and listen: the aliased version has the 13 folded partials, the dropped one
has none of them.

In [ ]:
dropped = asp.synthesize(events, sr=sr_low)
aliased = asp.synthesize(events, sr=sr_low, nyquist="alias")
asp.nb.play(dropped, sr_low, normalize=True)
asp.nb.play(aliased, sr_low, normalize=True)
asp.nb.show(asp.viz.plot_spectrum(asp.spectrum(aliased, sr_low)))

## Write audio

`write_audio` also refuses samples above full scale; `normalize=-1.0` first
scales the render to a peak of −1.0 dBFS.

In [ ]:
Path("rendered_audio").mkdir(exist_ok=True)
path = asp.write_audio("rendered_audio/conga_multidft.wav", audio, sr, normalize=-1.0)
print(path)